## Setup

In [13]:
import pandas as pd
# print(pd.__version__)
from sqlalchemy import create_engine, event, text
import pyodbc
pyodbc.drivers()

['SQL Server',
 'ODBC Driver 17 for SQL Server',
 'ODBC Driver 18 for SQL Server',
 'Microsoft Access Driver (*.mdb, *.accdb)',
 'Microsoft Excel Driver (*.xls, *.xlsx, *.xlsm, *.xlsb)',
 'Microsoft Access Text Driver (*.txt, *.csv)',
 'Microsoft Access dBASE Driver (*.dbf, *.ndx, *.mdx)']

In [2]:
# conda deactivate
# conda activate pigment2
# Get-Process -Name "python" | Stop-Process -Force

From now on, whenever the kernel freezes and VSCode buttons **don't** work:

Open PowerShell:
- Run: `C:\Users\thaop\anaconda3\shell\condabin\conda-hook.ps1`
- Run:` & "C:\Users\thaop\Desktop\fix_kernel.ps1"`
- Restart kernel in VSCode

Or more correct **THIS**:
```
(C:\Users\thaop\anaconda3\shell\condabin\conda-hook.ps1)
& "C:\Users\thaop\Desktop\fix_kernel.ps1"
```

Now go test your notebook — run several cells and tell me if it's working cleanly.

## Load Dataset

In [3]:
clean1 = pd.read_csv("../../semi_cleaned_data/cleaned_data_v2.csv", encoding='utf-8-sig')
clean2 = pd.read_csv("../../clean_data/ulta_clean_blush_v1.csv", encoding='utf-8-sig')

## Comparing Datasets
I just want to confirm which dataset should be used to convert into database.

In [4]:
print(f"Number of rows between clean1({len(clean1)}) and clean2({len(clean2)})")
print(f"{'Clean1 has more rows' if len(clean1)>len(clean2) else 'Clean2 has more rows' if len(clean1)>len(clean2) else "Equal!"}")

Number of rows between clean1(1514) and clean2(1514)
Equal!


In [5]:
print(f"Number of columns between clean1({len(clean1.columns)}) and clean2({len(clean2.columns)})")
print(f"The same columns in both datasets: ")

Number of columns between clean1(11) and clean2(13)
The same columns in both datasets: 


In [ ]:
# Get column names by putting into `set` containers
cols1 = set(clean1.columns)
cols2 = set(clean2.columns)

# Common columns
common_cols = cols1.intersection(cols2)
print(f"Common columns ({len(common_cols)}):")
print(sorted(common_cols))

# Columns only in clean1
only_clean1 = cols1 - cols2
print(f"\nOnly in clean1 ({len(only_clean1)}):")
print(sorted(only_clean1))

# Columns only in clean2
only_clean2 = cols2 - cols1
print(f"\nOnly in clean2 ({len(only_clean2)}):")
print(sorted(only_clean2))

Common columns (10):
['brand', 'description', 'price', 'product_name', 'shade', 'shade_url', 'standard_unit', 'standard_value', 'swatch_alt', 'swatch_img_url']

Only in clean1 (1):
['currency']

Only in clean2 (3):
['product_id', 'product_image_url', 'shade_id']


In [10]:
# Create comparison table
all_cols = sorted(list(cols1.union(cols2)))
comparison = pd.DataFrame({
    'column': all_cols,
    'in_clean1': [col in cols1 for col in all_cols],
    'in_clean2': [col in cols2 for col in all_cols]
})

print(comparison)

               column  in_clean1  in_clean2
0               brand       True       True
1            currency       True      False
2         description       True       True
3               price       True       True
4          product_id      False       True
5   product_image_url      False       True
6        product_name       True       True
7               shade       True       True
8            shade_id      False       True
9           shade_url       True       True
10      standard_unit       True       True
11     standard_value       True       True
12         swatch_alt       True       True
13     swatch_img_url       True       True


In [11]:
print(f"Total unique columns across both: {len(cols1.union(cols2))}")
print(f"clean1 columns: {len(cols1)}")
print(f"clean2 columns: {len(cols2)}")
print(f"Common columns: {len(common_cols)}")
print(f"Unique to clean1: {len(only_clean1)}")
print(f"Unique to clean2: {len(only_clean2)}")

Total unique columns across both: 14
clean1 columns: 11
clean2 columns: 13
Common columns: 10
Unique to clean1: 1
Unique to clean2: 3


**Conclusion**
After comparing both dataframes, the data set, which was assigned as `clean2`, is cleaner

## To Database

In [14]:
database_name = input("Please enter the name of the database: ") # ulta_blushes is the database name that we're going on in this notebook

In [15]:
engine = create_engine(
    f"mssql+pyodbc://LAPTOP-DU4JGOHJ/{database_name}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
    "&trusted_connection=yes"
    "&TrustServerCertificate=yes"
)

### wipe_table_clean()

The function can be use to wipe the whole table clean and still keeps the table's structure (columns and data types) without droppping the whole table in the database.

In [17]:
def wipe_tables_clean(engine):
    """Clears all data from tables in child-to-parent order to respect FK constraints."""
    tables = [
        'blushes'
              ]
    with engine.connect() as connection:
        trans = connection.begin()
        try:
            for table in tables:
                print(f"Clearing: {table}...")
                connection.execute(text(f"DELETE FROM {table}"))
            trans.commit()
            print("--- All tables cleared. ---")
        except Exception as e:
            trans.rollback()
            print(f"Error: {e}")

wipe_tables_clean(engine)

Clearing: blushes...
--- All tables cleared. ---


>**MUST CREATE THE DATABASE BEFORE EXPORT THE DATA**

Run the [file](ulta_database_structure.sql) to create the database `ulta_blush`

### Export to Database

In [18]:
# 'blushes' is a table name, we're creating in the new database 'ulta_blushes'
clean2.to_sql('blushes', engine, if_exists='append', index=False)

65